# Il problema lineare dei minimi quadrati 

I minimi quadrati permettono di trovare una soluzione **approssimata** quando un sistema lineare non ammette una soluzione esatta. Sono inoltre alla base della regressione lineare e di numerosi metodi di analisi dei dati.

In questo notebook introdurremo gradualmente il problema, la sua interpretazione geometrica e i principali metodi numerici per risolverlo.

## Obiettivi

Alla fine del notebook sapremo:

- riconoscere un problema lineare ai minimi quadrati;
- interpretare residuo e soluzione geometricamente;
- derivare le equazioni normali;
- risolvere il problema mediante equazioni normali e SVD;
- comprendere esistenza, unicità e soluzione di norma minima;
- collegare i minimi quadrati ai modelli predittivi (in particolare la regressione polinomiale)

### 7.1. Da dove nasce il problema?

Consideriamo un sistema lineare

$$
Ax=b, \qquad A\in\mathbb{R}^{m\times n},
\quad x\in\mathbb{R}^n, \quad b\in\mathbb{R}^m.
$$

Se $m>n$, ci sono più equazioni che incognite e il sistema si dice **sovradeterminato**. In presenza di dati reali, rumore o errori di misura, in generale non esiste un vettore $x$ che soddisfi esattamente tutte le equazioni.

Non vogliamo quindi risolvere esattamente $Ax=b$, ma trovare il vettore $x$ per cui $Ax$ è il più vicino possibile a $b$.

*Residuo e funzione obiettivo*

Per ogni possibile vettore $x$ definiamo il **residuo**

$$
r(x)=b-Ax.
$$

Il residuo misura la parte di $b$ che il modello $Ax$ non riesce a spiegare. Il problema lineare dei minimi quadrati è

$$
\boxed{x^*=\operatorname*{argmin}_{x\in\mathbb{R}^n}
\|b-Ax\|_2^2}.
$$

Poiché

$$
\|r(x)\|_2^2=r_1(x)^2+\cdots+r_m(x)^2,
$$

stiamo minimizzando la **somma dei quadrati degli errori**.

*Perché si usano i quadrati?*

L'uso dei quadrati ha alcune proprietà convenienti:

- gli errori positivi e negativi non si cancellano;
- gli errori grandi vengono penalizzati maggiormente;
- la funzione obiettivo è continua e derivabile;
- nel caso lineare il problema conduce a strumenti di algebra lineare efficienti.

Lo svantaggio è che un valore anomalo può avere un'influenza molto forte proprio perché il suo errore viene elevato al quadrato (rivedi lezione Prof. Morotti del 6/10/2026)

In [36]:
import numpy as np
import matplotlib.pyplot as plt

# Quattro equazioni e due incognite
A = np.array([[1., 0.],
              [1., 1.],
              [1., 2.],
              [1., 3.]])
b = np.array([1., 2., 2., 4.])

x_star, residui, rango, valori_singolari = np.linalg.lstsq(A, b, rcond=None)
b_stimato = A @ x_star
r = b-b_stimato

print('Soluzione LSQ:', np.round(x_star, 4))
print('Ax*:', np.round(b_stimato, 4))
print('b:  ', b)
print('Residuo b-Ax*:', np.round(r, 4))
print('Somma dei quadrati:', np.dot(r, r))

Soluzione LSQ: [0.9 0.9]
Ax*: [0.9 1.8 2.7 3.6]
b:   [1. 2. 2. 4.]
Residuo b-Ax*: [ 0.1  0.2 -0.7  0.4]
Somma dei quadrati: 0.6999999999999996


 ### 7.2 Esistenza e unicità

Il problema lineare ai minimi quadrati ammette sempre almeno una soluzione. L'unicità dipende dal rango di $A$:

- se $\operatorname{rank}(A)=n$, le colonne sono linearmente indipendenti e la soluzione è unica;
- se $\operatorname{rank}(A)<n$, esistono infinite soluzioni che producono lo stesso vettore $Ax^*$ e lo stesso residuo minimo.

Nel secondo caso è possibile scegliere, tra tutte le soluzioni, quella con norma euclidea minima.

### Interpretazione geometrica ed equazioni normali

Il vettore $Ax$ è una combinazione lineare delle colonne di $A$. Perciò, al variare di $x$, il vettore $Ax$ appartiene sempre allo spazio generato dalle colonne di $A$, indicato con

$$
\operatorname{Im}(A)=\operatorname{span}\{a_1,\ldots,a_n\}.
$$

Se $b$ non appartiene a questo sottospazio, l'equazione $Ax=b$ non ha soluzione. La soluzione ai minimi quadrati sceglie $Ax^*$ come il punto di $\operatorname{Im}(A)$ più vicino a $b$.

Quindi $Ax^*$ è la **proiezione ortogonale** di $b$ sullo spazio delle colonne di $A$.

![](immagini_sorgente/fig_minimi_proiezione.png)

<!-- ```{figure} immagini_sorgente/fig_minimi_proiezione.png
---
width: 100%
align: center
---
```

<p align="center">
  <img src="immagini_sorgente/fig_minimi_proiezione.png" width="700">
</p> -->



Nel punto più vicino, il residuo

$$
r^*=b-Ax^*
$$

è ortogonale a tutte le colonne di $A$. Pertanto

$$
A^Tr^*=0.
$$

Sostituendo $r^*=b-Ax^*$ si ottiene

$$
A^T(b-Ax^*)=0
\quad\Longleftrightarrow\quad
A^TAx^*=A^Tb.
$$

Questo sistema si dice **sistema delle equazioni normali**.

In [37]:
print('A^T r =', np.round(A.T @ r, 12))
print('||A^T r||_2 =', np.linalg.norm(A.T @ r))

# La matrice di proiezione sullo spazio delle colonne di A
P = A @ np.linalg.solve(A.T @ A, A.T)
print('||Pb-Ax*||_2 =', np.linalg.norm(P @ b-b_stimato))
print('||P^2-P||_2 =', np.linalg.norm(P @ P-P))

A^T r = [0. 0.]
||A^T r||_2 = 3.820199830714189e-15
||Pb-Ax*||_2 = 1.4217791915866692e-15
||P^2-P||_2 = 2.785946459622037e-16


<!-- ### 7.3 Equazioni normali

La stessa condizione può essere ottenuta minimizzando la funzione

$$
f(x)=\|b-Ax\|_2^2.
$$

Sviluppando il prodotto scalare:

$$
f(x)=x^TA^TAx-2x^TA^Tb+b^Tb.
$$

Il gradiente è

$$
\nabla f(x)=2A^T(Ax-b).
$$

La matrice Hessiana è $\nabla^2f(x)=2A^TA$ ed è semidefinita positiva (cioè i suoi autovalori sono positivi o nulli). La funzione è quindi convessa: ogni punto che annulla il gradiente è un minimo globale. Se $A$ ha rango massimo per colonne, l'Hessiana è definita positiva e il minimo è unico.

Imponendo $\nabla f(x)=0$ otteniamo il sistema delle **equazioni normali**:

$$
\boxed{A^TAx=A^Tb}.
$$ -->



La matrice $A^TA$ è sempre simmetrica e semidefinita positiva, perché

$$
z^TA^TAz=\|Az\|_2^2\geq0.
$$

Se $A$ ha rango massimo per colonne, allora $A^TA$ è definita positiva e invertibile. Le equazioni normali hanno quindi un'unica soluzione.

Poiché $A^TA$ è simmetrica definita positiva, il sistema può essere risolto con la fattorizzazione di Cholesky

$$
A^TA=LL^T,
$$

risolvendo prima $Ly=A^Tb$ e poi $L^Tx=y$.

In [38]:
# Soluzione mediante equazioni normali e Cholesky
G = A.T @ A
c = A.T @ b
L = np.linalg.cholesky(G)
y = np.linalg.solve(L, c)
x_normali = np.linalg.solve(L.T, y)

print('A^T A =\n', G)
print('Soluzione:', np.round(x_normali, 6))
print('Confronto con lstsq:', np.linalg.norm(x_normali-x_star))

A^T A =
 [[ 4.  6.]
 [ 6. 14.]]
Soluzione: [0.9 0.9]
Confronto con lstsq: 1.4812222630807097e-15


### Risoluzione di un problema di minimi quadrati con SVD


Vogliamo determinare una soluzione del problema

$$
\min_{x\in\mathbb{R}^n}\|Ax-b\|_2^2,
$$

dove $A\in\mathbb{R}^{m\times n}$ e $b\in\mathbb{R}^m$. La dimostrazione vale per matrici rettangolari e anche quando $A$ non ha rango pieno. Minimizziamo il quadrato della norma perché ha gli stessi minimizzatori della norma.


Ricordiamo che essendo $U$ matrice ortogonale, per ogni $z\in\mathbb{R}^m$ si ha

$$
\|U^Tz\|_2^2=(U^Tz)^T(U^Tz)=z^TUU^Tz=z^Tz=\|z\|_2^2.
$$

Applicando questa proprietà al residuo $z=Ax-b$ e inserendo $VV^T=I_n$, otteniamo i passaggi della slide:

$$
\begin{aligned}
\|Ax-b\|_2^2
&=\|U^T(Ax-b)\|_2^2\\
&=\|U^TAx-U^Tb\|_2^2\\
&=\|U^TAVV^Tx-U^Tb\|_2^2.
\end{aligned}
$$

**Perché usiamo la SVD completa?** Qui $U$ è quadrata e ortogonale, quindi la trasformazione conserva l'intera norma del residuo. Con la sola matrice dei $k$ vettori singolari sinistri associati ai valori singolari positivi occorrerebbe aggiungere il contributo ortogonale alla loro immagine.



Poniamo

$$
y=V^Tx\in\mathbb{R}^n,\qquad g=U^Tb\in\mathbb{R}^m.
$$

Poiché $V$ è ortogonale, il cambio di variabili è invertibile e $x=Vy$: minimizzare rispetto a $x$ equivale a minimizzare rispetto a $y$.

Dalla SVD segue che

$$
U^TAV=U^T(U\Sigma V^T)V=\Sigma.
$$

Pertanto

$$
\boxed{\|Ax-b\|_2^2=\|\Sigma y-g\|_2^2.}
$$

Il vantaggio è che $\Sigma$ è diagonale rettangolare: le componenti incognite non sono più accoppiate.



Le componenti del vettore $\Sigma y\in\mathbb{R}^m$ sono

$$
(\Sigma y)_i=
\begin{cases}
\sigma_i y_i,&1\leq i\leq k,\\
0,&k<i\leq m.
\end{cases}
$$

Di conseguenza,

$$
\boxed{
\|\Sigma y-g\|_2^2
=\sum_{i=1}^{k}(\sigma_i y_i-g_i)^2
+\sum_{i=k+1}^{m}g_i^2.
}
$$

La prima somma dipende da $y_1,\ldots,y_k$; la seconda è costante rispetto a tutte le incognite. Inoltre, se $k<n$, le componenti $y_{k+1},\ldots,y_n$ non compaiono nella funzione da minimizzare.




Ogni termine della prima somma è non negativo. Poiché $\sigma_i>0$ per $i=1,\ldots,k$, possiamo annullarli tutti scegliendo

$$
\sigma_i y_i-g_i=0
\quad\Longrightarrow\quad
\boxed{y_i=\frac{g_i}{\sigma_i}=\frac{(U^Tb)_i}{\sigma_i}=\frac{u_i^Tb}{\sigma_i},\qquad i=1,\ldots,k.}
$$

Questa scelta raggiunge il minimo globale, perché resta solo la somma costante:

$$
\boxed{\min_x\|Ax-b\|_2^2=\sum_{i=k+1}^{m}g_i^2.}
$$

Quindi il valore minimo della **norma** del residuo è

$$
\min_x\|Ax-b\|_2=\left(\sum_{i=k+1}^{m}g_i^2\right)^{1/2}.
$$

*Precisazione sulla slide:* $g_i$ è lo scalare $(U^Tb)_i=u_i^Tb$, non l'intero vettore $U^Tb$.



Dato che $x=Vy=\sum_{i=1}^n y_i v_i$, tutte le soluzioni del problema di minimi quadrati sono

$$
\boxed{
x=\sum_{i=1}^{k}\frac{u_i^Tb}{\sigma_i}v_i
+\sum_{i=k+1}^{n}\alpha_i v_i,
\qquad \alpha_i\in\mathbb{R}.
}
$$

- Se $k=n$, la matrice ha rango pieno per colonne e la soluzione è **unica**: la seconda somma non è presente.
- Se $k<n$, esistono **infinite soluzioni**: i coefficienti $\alpha_i$ sono liberi.

Infatti, per $i>k$ si ha $Av_i=0$: aggiungere una combinazione di questi vettori modifica $x$, ma lascia invariati $Ax$ e il residuo.



Quando le soluzioni sono infinite, possiamo scegliere quella di norma euclidea minima. Poiché $V$ è ortogonale,

$$
\|x\|_2^2=\|y\|_2^2
=\sum_{i=1}^{k}\left(\frac{u_i^Tb}{\sigma_i}\right)^2
+\sum_{i=k+1}^{n}\alpha_i^2.
$$

La prima somma è fissata dalla condizione di minimo del residuo. Per minimizzare anche la norma della soluzione dobbiamo porre $\alpha_i=0$ per ogni $i>k$. Otteniamo quindi

$$
\boxed{x^\dagger=\sum_{i=1}^{k}\frac{u_i^Tb}{\sigma_i}v_i.}
$$

Definiamo $\Sigma^\dagger\in\mathbb{R}^{n\times m}$ ponendo $1/\sigma_i$ nelle prime $k$ posizioni diagonali e zero altrove. La stessa formula si scrive

$$
x^\dagger=V\Sigma^\dagger U^Tb=A^\dagger b,
\qquad A^\dagger=V\Sigma^\dagger U^T.
$$

$A^\dagger$ è la **pseudoinversa di Moore–Penrose**. La soluzione $x^\dagger$ è sempre l'unica soluzione di norma minima tra tutti i minimizzatori; se $k=n$, coincide con l'unica soluzione del problema di minimi quadrati.

In [39]:
U, s, VT = np.linalg.svd(A, full_matrices=False)
tol = max(A.shape) * np.finfo(float).eps * s[0]
s_inv = np.array([1/sigma if sigma > tol else 0.0 for sigma in s])
A_piu = VT.T @ np.diag(s_inv) @ U.T
x_svd = A_piu @ b

print('Valori singolari:', np.round(s, 6))
print('Soluzione SVD:', np.round(x_svd, 6))
print('||x_SVD-x_lstsq||_2 =', np.linalg.norm(x_svd-x_star))

Valori singolari: [4.10003  1.090757]
Soluzione SVD: [0.9 0.9]
||x_SVD-x_lstsq||_2 = 2.482534153247273e-16


**Confronto tra i metodi**

| Metodo | Vantaggi | Limiti |
|:--|:--|:--|
| Equazioni normali + Cholesky | Semplice e rapido | Forma $A^TA$ e ne eleva al quadrato il condizionamento |
| SVD | Molto robusta; identifica rango e soluzione di norma minima | Più costosa |



 **Riepilogo**

Il problema lineare ai minimi quadrati

$$
\min_x\|b-Ax\|_2^2
$$

cerca il vettore $Ax$ più vicino a $b$ nello spazio generato dalle colonne di $A$. Il residuo ottimo è ortogonale a tale spazio.

Le equazioni normali sono semplici ma possono peggiorare il condizionamento. La SVD è più robusta e permette di trattare anche matrici di rango non massimo, producendo la soluzione di norma minima.

Nell'analisi dei dati, la stessa struttura matematica permette di stimare i coefficienti della regressione lineare.